# Indexing Pipeline

In [2]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

load_dotenv()

PDF_PATH = "docs/profile.pdf"
PERSIST_DIR = os.getenv("CHROMA_PERSIST_DIR", "./chroma_db")
COLLECTION = os.getenv("CHROMA_COLLECTION", "pdf_rag")


def load_pdf_pages(path: str) -> list[Document]:
    """Read every page of a PDF and return one Document per page."""
    
    loader = PyPDFLoader(PDF_PATH)
    documents = loader.load()
    return documents


# 1. Load PDF
documents = load_pdf_pages(PDF_PATH)
print(f"Loaded {len(documents)} pages")

# 2. Split documents into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100
)
chunks = splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks")

# 3. Create the embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# 4. Give every chunk a stable ID (filename + chunk number)
ids = [f"{os.path.basename(PDF_PATH)}-{i}" for i in range(len(chunks))]

# 5. Embed the chunks and store them in Chroma
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    ids=ids,
    collection_name=COLLECTION,
    persist_directory=PERSIST_DIR,
    collection_metadata={"hnsw:space": "cosine"}
)

print(f"Stored {len(chunks)} chunks in collection '{COLLECTION}'")


C:\Users\jason\AppData\Local\Temp\ipykernel_3016\1913169162.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 3 pages
Created 8 chunks


c:\Asadullah AI\Agentic-AI-Engineering-Course\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 828.14it/s]


Stored 8 chunks in collection 'pdf_rag'


# Retreival and Generating Pipeline

In [3]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

# 1. Initialize the embedding model (must match ingest.py)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# 2. Connect to the existing Chroma collection
vector_store = Chroma(
    collection_name=os.getenv("CHROMA_COLLECTION", "pdf_rag"),
    embedding_function=embeddings,
    persist_directory=os.getenv("CHROMA_PERSIST_DIR", "./chroma_db")
)

# 3. Create the retriever
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

# 4. Initialize the LLM
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

# 5. Create the prompt
prompt = ChatPromptTemplate.from_template("""
You are a helpful AI assistant.

Answer the question using ONLY the provided context.

If the answer is not available in the context,
say that you do not have enough information.

Context:
{context}

Question:
{question}

Answer:
""")

# 6. Create the RAG chain
rag_chain = prompt | llm | StrOutputParser()


def format_docs(docs):
    """Join chunks into one context string, tagging each with its page."""
    return "\n\n".join(
        f"[Page {doc.metadata.get('page', 0) + 1}] {doc.page_content}"
        for doc in docs
    )


def print_sources(docs):
    """Print each unique (file, page) pair once."""
    seen = set()
    print("SOURCES")
    for doc in docs:
        source = os.path.basename(doc.metadata.get("source", "unknown"))
        page = doc.metadata.get("page", 0) + 1
        if (source, page) in seen:
            continue
        seen.add((source, page))
        print(f"  {source} — Page {page}")


print("PDF Question Answering (type 'exit' to quit)\n")

while True:
    question = input("You: ").strip()

    if question.lower() in {"exit", "quit"}:
        break
    if not question:
        continue

    docs = retriever.invoke(question)
    context = format_docs(docs)

    answer = rag_chain.invoke({
        "context": context,
        "question": question
    })

    print(f"\nAI Assistant: {answer}\n")
    print_sources(docs)
    print()


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 755.65it/s]


PDF Question Answering (type 'exit' to quit)


AI Assistant: Asadullah AI

SOURCES
  profile.pdf — Page 1
  profile.pdf — Page 2


AI Assistant: The channel link is **asadullah‑ai.vercel.app**.

SOURCES
  profile.pdf — Page 2
  profile.pdf — Page 1


AI Assistant: Asadullah AI teaches how to build production‑ready AI systems. The channel’s tutorials cover:

* The theory and research behind **agentic AI** (“why it works”) and the detailed internals (“how it works”)  
* Building projects **from scratch** rather than copying code  
* Practical topics such as **AI automation with n8n**, setting up the open‑source **OmniRoute** AI router, **computer‑vision fine‑tuning**, and **MLOps**  

All lessons focus on understanding why a method works, how it works, and how to implement it yourself.

SOURCES
  profile.pdf — Page 3
  profile.pdf — Page 2
  profile.pdf — Page 1

